In [81]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
import matplotlib.pyplot as plt

In [82]:
data_2018 = pd.read_csv('Data/f1_2018_laps.csv')
data_2019 = pd.read_csv('Data/f1_2019_laps.csv')
data_2020 = pd.read_csv('Data/f1_2020_laps.csv')
data_2021 = pd.read_csv('Data/f1_2021_laps.csv')
data_2022 = pd.read_csv('Data/f1_2022_laps.csv')
data_2023 = pd.read_csv('Data/f1_2023_laps.csv')
data_2024 = pd.read_csv('Data/f1_2024_laps.csv')

In [87]:
def preprocess_data(data, test_race, categorical_features=['Driver', 'Team', 'Status']):
    agg_dict = {
        'Grid': 'first',
        'LapTime': 'mean',
        'LapNumber': 'mean',
        'SpeedFL': 'mean',
        'Sector1Time': 'mean',
        'Sector2Time': 'mean',
        'Sector3Time': 'mean',
        'PitStop': 'count',
        'Position': 'first',
        'Finish_Position': 'first',
        'Status': 'first',
        'AirTemp': 'first',
        'TrackTemp': 'first',
        'Humidity': 'first',
        'Rainfall': 'first',
        'WindSpeed': 'first',
        'Race_ID': 'first',
        'DNF': 'first',
        'Top10': 'first',
        'Grid_Finish_Delta': 'first',
        'Season': 'first'
    }

    compressed_data = data.groupby(['Driver', 'Team', 'Race']).agg(agg_dict).reset_index()
    compressed_data.dropna(inplace=True)
    compressed_data['Team'] = compressed_data['Team'].replace('Force India', 'Racing Point')
    train_df = compressed_data[compressed_data['Race'] != test_race].reset_index(drop=True)
    test_df = compressed_data[compressed_data['Race'] == test_race].reset_index(drop=True)
    train_df = pd.get_dummies(train_df, columns=categorical_features)
    test_df = pd.get_dummies(test_df, columns=categorical_features)
    test_df = test_df.reindex(columns=train_df.columns, fill_value=0)
    X_train = train_df.drop(['Finish_Position', 'Race', 'Race_ID'], axis=1)
    y_train = train_df['Finish_Position']
    X_test = test_df.drop(['Finish_Position', 'Race', 'Race_ID'], axis=1)
    y_test = test_df['Finish_Position']
    return X_train, y_train, X_test, y_test

def train_model(X_train, y_train, hidden_layer_sizes=(100,50), max_iter=500, random_state=42):
    model = MLPRegressor(hidden_layer_sizes=hidden_layer_sizes, max_iter=max_iter, random_state=random_state)
    model.fit(X_train, y_train)
    return model

def predict_and_rank(model, X_test):
    y_pred_raw = model.predict(X_test)
    y_pred_ranked = pd.Series(y_pred_raw).rank(method='first').astype(int)
    return y_pred_raw, y_pred_ranked

def compute_position_accuracies(y_test, y_pred_ranked):
    y_test_int = y_test.astype(int)
    exact_match = (y_test_int == y_pred_ranked).mean()
    top_2 = ((y_pred_ranked - y_test_int).abs() <= 1).mean()
    top_3 = ((y_pred_ranked - y_test_int).abs() <= 2).mean()
    return {
        'Exact_Position_Accuracy': exact_match,
        'Top2_Accuracy': top_2,
        'Top3_Accuracy': top_3
    }

def run_race_prediction_pipeline(data):
    test_race_id = data['Race_ID'].max()
    test_race_name = data.loc[data['Race_ID'] == test_race_id, 'Race'].iloc[0]
    X_train, y_train, X_test, y_test = preprocess_data(data, test_race_name)
    model = train_model(X_train, y_train)
    y_pred_raw, y_pred_ranked = predict_and_rank(model, X_test)
    accuracies = compute_position_accuracies(y_test, y_pred_ranked)
    comparison_df = pd.DataFrame({
        'Actual_Finish_Position': y_test.astype(int),
        'Predicted_Finish_Position': y_pred_ranked,
        'Predicted_Finish_Position_Raw': y_pred_raw
    }).sort_values(by='Actual_Finish_Position').reset_index(drop=True)
    return model, accuracies, comparison_df

In [88]:
model_2018, acc_2018, comp_2018 = run_race_prediction_pipeline(data_2018)
model_2019, acc_2019, comp_2019 = run_race_prediction_pipeline(data_2019)
model_2020, acc_2020, comp_2020 = run_race_prediction_pipeline(data_2020)
model_2021, acc_2021, comp_2021 = run_race_prediction_pipeline(data_2021)
model_2022, acc_2022, comp_2022 = run_race_prediction_pipeline(data_2022)
model_2023, acc_2023, comp_2023 = run_race_prediction_pipeline(data_2023)
model_2024, acc_2024, comp_2024 = run_race_prediction_pipeline(data_2024)

In [89]:
years = [2018, 2019, 2020, 2021, 2022, 2023, 2024]
data_list = [data_2018, data_2019, data_2020, data_2021, data_2022, data_2023, data_2024]

results = []

for year, data in zip(years, data_list):
    model, acc, comp = run_race_prediction_pipeline(data)
    results.append((year, acc, comp))

summary = pd.DataFrame([
    {
        'Year': year,
        'Exact_Position_Accuracy': acc['Exact_Position_Accuracy'],
        'Top3_Accuracy': acc['Top2_Accuracy'],
        'Top5_Accuracy': acc['Top3_Accuracy']
    }
    for year, acc, _ in results
])


In [90]:
def save_df_as_png(df, filename, title=None, figsize=(8, 8), fontsize=8):
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    
    if title:
        ax.set_title(title, fontsize=fontsize+2, fontweight='bold', pad=1)
    
    table = ax.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(1, 1.5) 
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

summary_percent = summary.copy()
accuracy_cols = ['Exact_Position_Accuracy', 'Top3_Accuracy', 'Top5_Accuracy']
for col in accuracy_cols:
    summary_percent[col] = (summary_percent[col] * 100).round(2).astype(str) + '%'

summary_filename = "race_accuracy_summary.png"
save_df_as_png(summary_percent, summary_filename, title="Race Prediction Accuracy Summary (%)")

for year, acc, comp in results:
    filename = f"comparison_{year}.png"
    save_df_as_png(comp, filename, title=f"Year {year} - Actual vs Predicted Positions")
